In [1]:
import pandas as pd
import os

RAW_DIR = "../data/raw"

train = pd.read_parquet(os.path.join(RAW_DIR, "chicago_2015_2024.parquet"))
test  = pd.read_parquet(os.path.join(RAW_DIR, "chicago_2025.parquet"))

train.shape, test.shape


((2477274, 5), (236404, 5))

In [2]:
train.dtypes


id                      object
date            datetime64[ns]
primary_type            object
latitude               float64
longitude              float64
dtype: object

In [3]:
train.isna().mean().sort_values(ascending=False)


id              0.0
date            0.0
primary_type    0.0
latitude        0.0
longitude       0.0
dtype: float64

In [4]:
train["id"].duplicated().sum()


np.int64(551)

In [5]:
train = train.drop_duplicates(subset="id")


In [6]:
train = train.dropna(subset=["date", "latitude", "longitude"])
test  = test.dropna(subset=["date", "latitude", "longitude"])


In [7]:
train["date"] = pd.to_datetime(train["date"])
test["date"]  = pd.to_datetime(test["date"])


In [8]:
train["primary_type"].value_counts().head(10)


primary_type
THEFT                  557403
BATTERY                459765
CRIMINAL DAMAGE        277724
ASSAULT                200941
DECEPTIVE PRACTICE     166729
OTHER OFFENSE          157956
MOTOR VEHICLE THEFT    144030
NARCOTICS              100694
BURGLARY               100263
ROBBERY                 95906
Name: count, dtype: int64

In [9]:
CRIME_TYPE = "THEFT"

train = train[train["primary_type"] == CRIME_TYPE]
test  = test[test["primary_type"] == CRIME_TYPE]

train.shape, test.shape


((557403, 5), (55173, 5))

In [10]:
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

train.to_parquet(os.path.join(PROCESSED_DIR, "chicago_clean_2015_2024.parquet"), index=False)
test.to_parquet(os.path.join(PROCESSED_DIR, "chicago_clean_2025.parquet"), index=False)

print("Clean data saved")


Clean data saved


In [11]:
import pandas as pd

RAW_TRAIN_PATH = "../data/raw/chicago_2015_2024.parquet"
RAW_TEST_PATH  = "../data/raw/chicago_2025.parquet"

CLEAN_TRAIN_PATH = "../data/processed/chicago_clean_2015_2024.parquet"
CLEAN_TEST_PATH  = "../data/processed/chicago_clean_2025.parquet"

raw_train = pd.read_parquet(RAW_TRAIN_PATH)
raw_test  = pd.read_parquet(RAW_TEST_PATH)

clean_train = pd.read_parquet(CLEAN_TRAIN_PATH)
clean_test  = pd.read_parquet(CLEAN_TEST_PATH)

summary = pd.DataFrame({
    "dataset": ["train", "test"],
    "raw_rows": [len(raw_train), len(raw_test)],
    "clean_rows": [len(clean_train), len(clean_test)],
})
summary["rows_removed"] = summary["raw_rows"] - summary["clean_rows"]
summary["retention_rate"] = (summary["clean_rows"] / summary["raw_rows"]).round(4)

summary


,dataset,raw_rows,clean_rows,rows_removed,retention_rate
0,train,2477274,557403,1919871,0.2250
1,test,236404,55173,181231,0.2334


In [ ]:
def removal_breakdown(raw_df, clean_df):
    dup_ids = raw_df["id"].duplicated().sum()
    missing_loc = raw_df[["latitude", "longitude"]].isna().any(axis=1).sum()
    missing_date = raw_df["date"].isna().sum()


    return pd.Series({
        "raw_rows": len(raw_df),
        "clean_rows": len(clean_df),
        "dup_id_count_in_raw": int(dup_ids),
        "missing_location_in_raw": int(missing_loc),
        "missing_date_in_raw": int(missing_date),
    })

breakdown = pd.DataFrame({
    "train": removal_breakdown(raw_train, clean_train),
    "test":  removal_breakdown(raw_test, clean_test),
}).T

breakdown


,raw_rows,clean_rows,dup_id_count_in_raw,missing_location_in_raw,missing_date_in_raw
train,2477274,557403,551,0,0
test,236404,55173,2407,0,0
